# CIC-IDS2017 — DQN Reinforcement Learning Pipeline
## Fixed & Optimised | No Identifier Leakage | Improved Reward Shaping

**Project:** Graduation Project — RL vs Static ML Firewall Comparison  
**Phase 1:** Train DQN agent on CIC-IDS2017, compare with RF & GB  
**Phase 2 (next notebook):** Test all three models on NSL-KDD (cross-environment generalisation)

> **Fixes applied:**
> - `source_ip`, `destination_ip`, `flow_id`, `source_port`, `timestamp` removed (same as ML notebook)
> - FPR now reported in evaluation
> - Reward shaped to penalise **missed attacks more than false positives** (correct for firewall)
> - `N_EPISODES` guidance added — 20 episodes recommended for convergence
> - `X_train` / `X_test` now passed as numpy arrays consistently to DQN


## 1 · System Check

In [3]:
import platform, psutil, os

print("=" * 55)
print("  SYSTEM INFORMATION")
print("=" * 55)
print(f"  OS     : {platform.system()} {platform.release()}")
print(f"  CPU    : {os.cpu_count()} logical cores")
ram = psutil.virtual_memory()
print(f"  RAM    : {ram.total/1e9:.1f} GB total  |  {ram.available/1e9:.1f} GB free")

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  PyTorch: {torch.__version__}")
print(f"  Device : {device}  {'(GPU — fast!)' if device.type=='cuda' else '(CPU — fine for this)'}")
print("=" * 55)


  SYSTEM INFORMATION
  OS     : Windows 11
  CPU    : 8 logical cores
  RAM    : 16.9 GB total  |  3.3 GB free
  PyTorch: 2.9.1+cpu
  Device : cpu  (CPU — fine for this)


## 2 · Imports

In [5]:
import pandas as pd
import numpy as np
import warnings, pickle, time, json, gc, random
from pathlib import Path
from collections import deque
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    precision_score, recall_score, matthews_corrcoef
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ All imports OK  |  device = {device}")


✓ All imports OK  |  device = cpu


## 3 · Data Loading — Same Stratified 10 % Sample

> We load **exactly the same data** as the ML notebook  
> (same `DATA_DIR`, same `SAMPLE_FRAC`, same `RANDOM_STATE = 42`).  
> This is what makes the comparison between RL and RF/GB fair.


In [7]:
# ── ① Change this to your folder (same as ML notebook) ───────────────────────
DATA_DIR    = Path(r'C:/Users/DELL/Desktop/Graduation Project/GP_Models/data/TrafficLabelling')
SAMPLE_FRAC = 0.10
# ─────────────────────────────────────────────────────────────────────────────

csv_files = sorted(DATA_DIR.glob('*.csv'))
print(f"Found {len(csv_files)} CSV files")

chunks = []
for f in csv_files:
    try:
        tmp = pd.read_csv(f, encoding='cp1252', low_memory=False)
        tmp.columns = tmp.columns.str.strip()
        label_col = next((c for c in ['Label','label','Class','class']
                          if c in tmp.columns), tmp.columns[-1])
        sampled = (tmp.groupby(label_col, group_keys=False)
                      .apply(lambda x: x.sample(frac=SAMPLE_FRAC,
                                                 random_state=RANDOM_STATE)))
        chunks.append(sampled)
        del tmp; gc.collect()
        print(f"  ✓ {f.name}: {len(sampled):,} rows")
    except Exception as e:
        print(f"  ✗ {f.name}: {e}")

df = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()
print(f"\n✓ Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")


Found 8 CSV files
  ✓ Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 22,575 rows
  ✓ Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 28,647 rows
  ✓ Friday-WorkingHours-Morning.pcap_ISCX.csv: 19,104 rows
  ✓ Monday-WorkingHours.pcap_ISCX.csv: 52,992 rows
  ✓ Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 28,861 rows
  ✓ Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 17,037 rows
  ✓ Tuesday-WorkingHours.pcap_ISCX.csv: 44,591 rows
  ✓ Wednesday-workingHours.pcap_ISCX.csv: 69,270 rows

✓ Dataset: 283,077 rows × 85 columns


## 4 · Cleaning & Target Variable

In [9]:
from sklearn.preprocessing import LabelEncoder

# Identify label column
LABEL_COL = next((c for c in ['Label','label','Class','class']
                  if c in df.columns), df.columns[-1])

# ── Standardise column names ────────────────────────────────────────────────
df.columns = (df.columns.str.strip()
                .str.replace(' ', '_', regex=False)
                .str.lower())
LABEL_COL = LABEL_COL.strip().lower().replace(' ','_')

# ── Binary target BEFORE dropping label ─────────────────────────────────────
df['target'] = (~df[LABEL_COL].str.strip().str.upper()
                               .isin(['BENIGN','NORMAL','LEGITIMATE'])).astype(np.int8)
print(f'Benign  (0): {(df["target"]==0).sum():,}  ({(df["target"]==0).mean()*100:.1f}%)')
print(f'Attack  (1): {(df["target"]==1).sum():,}  ({(df["target"]==1).mean()*100:.1f}%)')

# ── Drop identifier columns + raw label ─────────────────────────────────────
# source_ip had 84.6% feature importance in previous run — pure leakage!
IDENTIFIER_COLS = [
    'flow_id', 'source_ip', 'destination_ip', 'src_ip', 'dst_ip',
    'sourceip', 'destip', 'timestamp', 'flowstarttime', 'flowendtime',
    'source_port', 'src_port',
    'fwd_header_length.1',
    LABEL_COL    # drop raw string label — target binary col already created
]
to_drop = [c for c in df.columns if c in IDENTIFIER_COLS]
df.drop(columns=to_drop, inplace=True, errors='ignore')
print(f'\n✓ Dropped identifier columns: {to_drop}')

# ── Handle inf / NaN ─────────────────────────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
print(f'✓ Replaced inf/NaN values with 0')

# ── Encode any remaining string columns ─────────────────────────────────────
obj_cols = [c for c in df.select_dtypes(include='object').columns if c != 'target']
for c in obj_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c].astype(str)).astype(np.int32)
if obj_cols:
    print(f'✓ Encoded remaining categorical columns: {obj_cols}')

# ── Sanity check: no string columns remain ───────────────────────────────────
remaining_obj = df.select_dtypes(include='object').columns.tolist()
if remaining_obj:
    print(f'⚠ Force-dropping remaining object columns: {remaining_obj}')
    df.drop(columns=remaining_obj, inplace=True)
else:
    print('✓ All columns numeric — safe for float32 conversion')

# ── Memory optimisation ──────────────────────────────────────────────────────
f64 = df.select_dtypes('float64').columns
i64 = df.select_dtypes('int64').columns
df[f64] = df[f64].astype(np.float32)
df[i64] = df[i64].astype(np.int32)

mem = df.memory_usage(deep=True).sum() / 1024**2
print(f'\nFinal shape: {df.shape}   RAM: {mem:.0f} MB')


Benign  (0): 227,311  (80.3%)
Attack  (1): 55,766  (19.7%)

✓ Dropped identifier columns: ['flow_id', 'source_ip', 'source_port', 'destination_ip', 'timestamp', 'fwd_header_length.1', 'label']
✓ Replaced inf/NaN values with 0
✓ All columns numeric — safe for float32 conversion

Final shape: (283077, 79)   RAM: 84 MB


## 5 · Train / Test Split & Normalisation

In [11]:
X = df.drop(columns=['target']).values.astype(np.float32)
y = df['target'].values.astype(np.int64)
FEATURE_NAMES = df.drop(columns=['target']).columns.tolist()
FEATURE_DIM   = X.shape[1]

print(f'Features : {FEATURE_DIM}  (behaviour features only, no identifiers)')
print(f'Samples  : {len(y):,}')
print(f'Attack rate: {y.mean()*100:.1f}%')

# Drop exact duplicate rows BEFORE splitting
df_temp = pd.DataFrame(X, columns=FEATURE_NAMES)
df_temp['target'] = y
n_before = len(df_temp)
df_temp = df_temp.drop_duplicates()
X = df_temp.drop(columns=['target']).values.astype(np.float32)
y = df_temp['target'].values.astype(np.int64)
print(f'After dedup: {len(y):,} rows  (removed {n_before - len(y):,} duplicates)')
del df_temp

# Stratified 70/30 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

# Verify no leakage
train_set = set(map(tuple, X_train[:100].astype(np.float16)))
test_set  = set(map(tuple, X_test[:100].astype(np.float16)))
print(f'Overlap check (should be 0): {len(train_set & test_set)}')

# Normalise — fit on TRAIN ONLY, apply to test
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype(np.float32)
X_test_sc  = scaler.transform(X_test).astype(np.float32)

# Keep as numpy arrays for the DQN (pytorch tensors built inside agent)
del X
gc.collect()
print(f'\nTrain : {len(X_train_sc):,}  |  Test : {len(X_test_sc):,}')
print('✓ Split and scaling done')


Features : 78  (behaviour features only, no identifiers)
Samples  : 283,077
Attack rate: 19.7%
After dedup: 268,174 rows  (removed 14,903 duplicates)
Overlap check (should be 0): 0

Train : 187,721  |  Test : 80,453
✓ Split and scaling done


## 6 · DQN Architecture

The network takes a **state vector** (one normalised network flow) and outputs  
**Q-values** for two actions: Allow (0) and Block (1).

Architecture follows the FireRL / MDPI DQN papers:  
Input → 256 → 128 → 64 → 2 outputs


In [13]:
class DQN(nn.Module):
    """
    Deep Q-Network for firewall policy.
    Input  : state vector  (1 × FEATURE_DIM)
    Output : Q-values      (1 × 2)  [Q_allow, Q_block]
    """
    def __init__(self, state_dim: int, action_dim: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        return self.net(x)


# Quick sanity check
dummy = torch.zeros(1, FEATURE_DIM)
model_test = DQN(FEATURE_DIM)
out = model_test(dummy)
print(f"✓ DQN architecture OK")
print(f"  Input  : {FEATURE_DIM} features")
print(f"  Output : {out.shape[1]} Q-values  [Q_allow, Q_block]")

total_params = sum(p.numel() for p in model_test.parameters())
print(f"  Params : {total_params:,}")
del model_test


✓ DQN architecture OK
  Input  : 78 features
  Output : 2 Q-values  [Q_allow, Q_block]
  Params : 61,506


## 7 · Experience Replay Buffer

In [16]:
class ReplayBuffer:
    """
    Stores (state, action, reward, next_state, done) transitions.
    Randomly samples mini-batches to break temporal correlations —
    a key technique in DQN (DeepMind, 2015).
    """
    def __init__(self, capacity: int = 50_000):
        self.buf = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buf.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buf, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (torch.FloatTensor(np.array(s)).to(device),
                torch.LongTensor(a).to(device),
                torch.FloatTensor(r).to(device),
                torch.FloatTensor(np.array(ns)).to(device),
                torch.FloatTensor(d).to(device))

    def __len__(self): return len(self.buf)


print("✓ ReplayBuffer ready  (capacity = 50,000 transitions)")


✓ ReplayBuffer ready  (capacity = 50,000 transitions)


## 8 · DQN Agent

Key design choices (all from your papers):
- **ε-greedy exploration** decaying from 1.0 → 0.05
- **Target network** updated every 200 steps (stabilises training)
- **Prioritised reward** +1 correct block / correct allow, −1 missed attack (false negative), −0.5 false alarm (false positive)


In [18]:
class DQNAgent:
    def __init__(self, state_dim, action_dim=2,
                 lr=1e-3, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 buffer_capacity=50_000, batch_size=256,
                 target_update_freq=200):

        self.action_dim         = action_dim
        self.gamma              = gamma
        self.epsilon            = epsilon_start
        self.epsilon_end        = epsilon_end
        self.epsilon_decay      = epsilon_decay
        self.batch_size         = batch_size
        self.target_update_freq = target_update_freq
        self.steps              = 0

        self.policy_net = DQN(state_dim, action_dim).to(device)
        self.target_net = DQN(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.memory    = ReplayBuffer(buffer_capacity)

    def select_action(self, state: np.ndarray) -> int:
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        with torch.no_grad():
            s = torch.FloatTensor(state).unsqueeze(0).to(device)
            return int(self.policy_net(s).argmax(dim=1).item())

    @staticmethod
    def compute_reward(action: int, true_label: int) -> float:
        """
        Asymmetric reward shaping — reflects real firewall priorities:
        Missing an attack is far worse than a false alarm.
        Based on FireRL (Yang et al., 2025) and ARCS (MDPI, 2025).
        
        +1.0  correct decision (TP or TN)
        -2.0  missed attack   (FN) — highest penalty, security-critical
        -0.5  false positive  (FP) — annoying but not catastrophic
        """
        if action == true_label:
            return +1.0
        elif action == 0 and true_label == 1:   # missed attack
            return -2.0
        else:                                    # false positive
            return -0.5

    def store(self, s, a, r, ns, done):
        self.memory.push(s, a, r, ns, done)

    def train_step(self) -> float:
        if len(self.memory) < self.batch_size:
            return 0.0

        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)

        q_values = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            max_next_q = self.target_net(next_states).max(1)[0]
            targets    = rewards + self.gamma * max_next_q * (1 - dones)

        loss = F.mse_loss(q_values, targets)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()

        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        return loss.item()


agent = DQNAgent(state_dim=FEATURE_DIM)
print('✓ DQN Agent ready')
print(f'  Policy net params : {sum(p.numel() for p in agent.policy_net.parameters()):,}')
print(f'  Learning rate     : 1e-3')
print(f'  Gamma (discount)  : 0.99')
print(f'  Epsilon start→end : 1.0 → 0.05  (decay 0.995)')
print(f'  Batch size        : 256')
print(f'  Target update     : every 200 steps')
print(f'  Reward: +1 correct | -2 missed attack | -0.5 false positive')


✓ DQN Agent ready
  Policy net params : 61,506
  Learning rate     : 1e-3
  Gamma (discount)  : 0.99
  Epsilon start→end : 1.0 → 0.05  (decay 0.995)
  Batch size        : 256
  Target update     : every 200 steps
  Reward: +1 correct | -2 missed attack | -0.5 false positive


## 9 · Training Loop

> **How this works:**  
> The agent sees one flow at a time (state), decides Allow/Block (action),  
> gets a reward signal, and stores the experience.  
> Every step it samples a random mini-batch and updates its neural network.  
> Over many episodes it learns which features indicate an attack.

> **⏱ Expected time:** ~5–15 min on CPU depending on your laptop.


In [20]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
# Recommended: 20 episodes for convergence on 10% CICIDS2017 sample
# Set to 5 for a quick test run first, then re-run with 20 for final results
N_EPISODES   = 20
LOG_INTERVAL = 2
# ─────────────────────────────────────────────────────────────────────────────

train_size = len(X_train_sc)
X_train_np = X_train_sc if isinstance(X_train_sc, np.ndarray) else X_train_sc.values

history = {'episode': [], 'loss': [], 'reward': [],
           'accuracy': [], 'f1': [], 'epsilon': [], 'fpr': []}

print(f'Training DQN for {N_EPISODES} episodes  |  {train_size:,} steps/episode')
print(f'Device: {device}')
print(f'Reward shaping: +1 correct | -2 missed attack | -0.5 false positive')
print('=' * 65)

t_start = time.time()

for episode in range(1, N_EPISODES + 1):
    ep_loss, ep_reward = 0.0, 0.0
    preds, trues       = [], []

    idx = np.random.permutation(train_size)

    for i in idx:
        state      = X_train_np[i]
        true_label = int(y_train[i])

        action = agent.select_action(state)
        reward = agent.compute_reward(action, true_label)

        next_state = X_train_np[(i + 1) % train_size]
        agent.store(state, action, float(reward), next_state, 1.0)

        loss = agent.train_step()

        ep_loss   += loss
        ep_reward += reward
        preds.append(action)
        trues.append(true_label)

    preds_arr = np.array(preds)
    trues_arr = np.array(trues)
    acc = np.mean(preds_arr == trues_arr)
    f1  = f1_score(trues_arr, preds_arr, zero_division=0)
    fpr_ep = (preds_arr[trues_arr==0]==1).sum() / max((trues_arr==0).sum(), 1)

    history['episode'].append(episode)
    history['loss'].append(ep_loss / train_size)
    history['reward'].append(ep_reward / train_size)
    history['accuracy'].append(acc)
    history['f1'].append(f1)
    history['epsilon'].append(agent.epsilon)
    history['fpr'].append(fpr_ep)

    elapsed   = time.time() - t_start
    remaining = elapsed / episode * (N_EPISODES - episode)

    if episode % LOG_INTERVAL == 0 or episode == 1:
        print(f'  Ep {episode:2d}/{N_EPISODES}  '
              f'Acc={acc:.4f}  F1={f1:.4f}  '
              f'FPR={fpr_ep:.4f}  '
              f'Reward={ep_reward/train_size:+.3f}  '
              f'ε={agent.epsilon:.3f}  '
              f'[{elapsed/60:.1f}m, ~{remaining/60:.1f}m left]')

print('=' * 65)
print(f'\n✓ Training complete in {(time.time()-t_start)/60:.1f} minutes')


Training DQN for 20 episodes  |  187,721 steps/episode
Device: cpu
Reward shaping: +1 correct | -2 missed attack | -0.5 false positive
  Ep  1/20  Acc=0.9572  F1=0.8887  FPR=0.0395  Reward=+0.920  ε=0.050  [23.5m, ~445.8m left]
  Ep  2/20  Acc=0.9600  F1=0.8958  FPR=0.0372  Reward=+0.926  ε=0.050  [47.9m, ~431.1m left]
  Ep  4/20  Acc=0.9644  F1=0.9060  FPR=0.0318  Reward=+0.932  ε=0.050  [97.3m, ~389.0m left]
  Ep  6/20  Acc=0.9583  F1=0.8919  FPR=0.0393  Reward=+0.923  ε=0.050  [146.8m, ~342.6m left]
  Ep  8/20  Acc=0.9616  F1=0.8991  FPR=0.0344  Reward=+0.927  ε=0.050  [196.4m, ~294.7m left]
  Ep 10/20  Acc=0.9640  F1=0.9051  FPR=0.0325  Reward=+0.932  ε=0.050  [246.4m, ~246.4m left]
  Ep 12/20  Acc=0.9611  F1=0.8982  FPR=0.0360  Reward=+0.927  ε=0.050  [295.9m, ~197.3m left]
  Ep 14/20  Acc=0.9577  F1=0.8909  FPR=0.0410  Reward=+0.924  ε=0.050  [345.7m, ~148.2m left]
  Ep 16/20  Acc=0.9644  F1=0.9063  FPR=0.0322  Reward=+0.933  ε=0.050  [395.5m, ~98.9m left]
  Ep 18/20  Acc=0.9620 

## 10 · Evaluation on Test Set

In [22]:
print('Evaluating DQN on held-out test set …')
agent.policy_net.eval()

X_test_np = X_test_sc if isinstance(X_test_sc, np.ndarray) else X_test_sc.values

preds_rl  = []
probas_rl = []

with torch.no_grad():
    batch_size_eval = 1024
    for start in range(0, len(X_test_np), batch_size_eval):
        batch  = torch.FloatTensor(X_test_np[start:start+batch_size_eval]).to(device)
        q_vals = agent.policy_net(batch)
        proba  = torch.softmax(q_vals, dim=1)[:, 1]
        pred   = q_vals.argmax(dim=1)
        preds_rl.extend(pred.cpu().numpy().tolist())
        probas_rl.extend(proba.cpu().numpy().tolist())

preds_rl  = np.array(preds_rl)
probas_rl = np.array(probas_rl)

acc_rl  = (preds_rl == y_test).mean()
prec_rl = precision_score(y_test, preds_rl, zero_division=0)
rec_rl  = recall_score(y_test, preds_rl, zero_division=0)
f1_rl   = f1_score(y_test, preds_rl, zero_division=0)
auc_rl  = roc_auc_score(y_test, probas_rl)
mcc_rl  = matthews_corrcoef(y_test, preds_rl)
fpr_rl  = (preds_rl[y_test==0]==1).sum() / (y_test==0).sum()

print('=' * 55)
print('  DQN RL AGENT — TEST SET PERFORMANCE')
print('  (Behaviour features only — no identifier leakage)')
print('=' * 55)
print(classification_report(y_test, preds_rl,
                             target_names=['Benign','Attack'], zero_division=0))
print(f'  Accuracy        : {acc_rl:.4f}')
print(f'  Precision       : {prec_rl:.4f}')
print(f'  Recall (DR)     : {rec_rl:.4f}')
print(f'  F1-Score        : {f1_rl:.4f}')
print(f'  ROC-AUC         : {auc_rl:.4f}')
print(f'  Matthews CC     : {mcc_rl:.4f}')
print(f'  False Pos. Rate : {fpr_rl:.4f}  ({fpr_rl*100:.2f}% of benign traffic blocked)')


Evaluating DQN on held-out test set …
  DQN RL AGENT — TEST SET PERFORMANCE
  (Behaviour features only — no identifier leakage)
              precision    recall  f1-score   support

      Benign       0.97      1.00      0.98     65867
      Attack       0.99      0.84      0.91     14586

    accuracy                           0.97     80453
   macro avg       0.98      0.92      0.95     80453
weighted avg       0.97      0.97      0.97     80453

  Accuracy        : 0.9696
  Precision       : 0.9897
  Recall (DR)     : 0.8411
  F1-Score        : 0.9093
  ROC-AUC         : 0.9963
  Matthews CC     : 0.8955
  False Pos. Rate : 0.0019  (0.19% of benign traffic blocked)


## 11 · Training Curves & Evaluation Plots

In [25]:
cm_rl = confusion_matrix(y_test, preds_rl)
cm_pct = cm_rl.astype(float) / cm_rl.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('DQN RL Agent — CIC-IDS2017 (Behaviour Features Only)',
             fontsize=14, fontweight='bold')

# 1 · Accuracy over episodes
ax = axes[0, 0]
ax.plot(history['episode'], history['accuracy'], marker='o', color='#3498db', lw=2)
ax.set_title('Accuracy over Episodes'); ax.set_xlabel('Episode')
ax.set_ylabel('Accuracy'); ax.set_ylim([0, 1.05]); ax.grid(alpha=0.3)

# 2 · F1 over episodes
ax = axes[0, 1]
ax.plot(history['episode'], history['f1'], marker='s', color='#2ecc71', lw=2)
ax.set_title('F1-Score over Episodes'); ax.set_xlabel('Episode')
ax.set_ylabel('F1'); ax.set_ylim([0, 1.05]); ax.grid(alpha=0.3)

# 3 · FPR over episodes (new!)
ax = axes[0, 2]
ax.plot(history['episode'], history['fpr'], marker='^', color='#e74c3c', lw=2)
ax.set_title('False Positive Rate over Episodes')
ax.set_xlabel('Episode'); ax.set_ylabel('FPR')
ax.set_ylim([0, max(history['fpr'])*1.2 + 0.01]); ax.grid(alpha=0.3)

# 4 · Confusion matrix with percentages
ax = axes[1, 0]
annot = np.array([[f'{v}\n({cm_pct[i,j]:.1f}%)' for j,v in enumerate(row)]
                   for i,row in enumerate(cm_rl)])
sns.heatmap(cm_rl, annot=annot, fmt='', cmap='Purples', ax=ax,
            xticklabels=['Benign','Attack'], yticklabels=['Benign','Attack'])
ax.set_title('Confusion Matrix (DQN)')
ax.set_ylabel('True label'); ax.set_xlabel('Predicted label')

# 5 · ROC curve
ax = axes[1, 1]
fpr_c, tpr_c, _ = roc_curve(y_test, probas_rl)
ax.plot(fpr_c, tpr_c, lw=2, color='purple', label=f'DQN AUC={auc_rl:.4f}')
ax.plot([0,1],[0,1],'--', color='grey', label='Random')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC Curve (DQN)')
ax.legend(); ax.grid(alpha=0.3)

# 6 · Epsilon decay
ax = axes[1, 2]
ax.plot(history['episode'], history['epsilon'], marker='D', color='#e67e22', lw=2)
ax.set_title('ε Decay (Exploration → Exploitation)')
ax.set_xlabel('Episode'); ax.set_ylabel('Epsilon'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('rl_training_results_fixed.png', dpi=100)
plt.show()
print('✓ Saved: rl_training_results_fixed.png')


✓ Saved: rl_training_results_fixed.png


## 12 · Phase 1 Comparison: RF vs GB vs DQN

In [27]:
META_PATH = Path('outputsRL/metadata.json')

if META_PATH.exists():
    with open(META_PATH) as fh:
        meta = json.load(fh)
    rf_f1  = meta['RF']['f1'];         rf_auc  = meta['RF']['roc_auc']
    rf_rec = meta['RF'].get('recall',  0.0);   rf_fpr = meta['RF'].get('fpr', 0.0)
    gb_f1  = meta['GB_tuned']['f1'];   gb_auc  = meta['GB_tuned']['roc_auc']
    gb_rec = meta['GB_tuned'].get('recall', 0.0); gb_fpr = meta['GB_tuned'].get('fpr', 0.0)
else:
    print('⚠ outputs/metadata.json not found — run the ML notebook first.')
    rf_f1=rf_auc=rf_rec=rf_fpr=gb_f1=gb_auc=gb_rec=gb_fpr=0.0

print('=' * 72)
print('  PHASE 1 COMPARISON — CIC-IDS2017 (behaviour features, no leakage)')
print('=' * 72)
print(f"  {'Model':<22}  {'F1':>7}  {'ROC-AUC':>8}  {'Recall':>7}  {'FPR':>7}")
print('-' * 72)
print(f"  {'Random Forest':<22}  {rf_f1:>7.4f}  {rf_auc:>8.4f}  {rf_rec:>7.4f}  {rf_fpr:>7.4f}")
print(f"  {'Gradient Boosting':<22}  {gb_f1:>7.4f}  {gb_auc:>8.4f}  {gb_rec:>7.4f}  {gb_fpr:>7.4f}")
print(f"  {'DQN RL Agent':<22}  {f1_rl:>7.4f}  {auc_rl:>8.4f}  {rec_rl:>7.4f}  {fpr_rl:>7.4f}")
print('=' * 72)
print('\n→ NEXT: Run Phase 2 notebook — test all three on NSL-KDD (no retraining)')

# Bar chart — 4 metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phase 1: All Models on CIC-IDS2017 (Behaviour Features Only)',
             fontsize=12, fontweight='bold')

models_list = ['Random Forest', 'Gradient Boosting', 'DQN (RL)']
colors = ['#3498db', '#2ecc71', '#9b59b6']
x = np.arange(len(models_list)); w = 0.2

ax = axes[0]
for idx, (mname, vals) in enumerate([
    ('F1-Score', [rf_f1,  gb_f1,  f1_rl]),
    ('Recall',   [rf_rec, gb_rec, rec_rl]),
]):
    bars = ax.bar(x + idx*w - w/2, vals, w, label=mname, color=['#3498db','#e74c3c'][idx])
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.003,
                f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(models_list, rotation=10)
ax.set_ylim([0.5, 1.05]); ax.set_title('F1 & Recall'); ax.legend(); ax.grid(alpha=0.3, axis='y')

ax = axes[1]
fprs = [rf_fpr, gb_fpr, fpr_rl]
bars = ax.bar(x, fprs, 0.5, color=colors)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.001,
            f'{b.get_height():.4f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(models_list, rotation=10)
ax.set_title('False Positive Rate (lower = better)')
ax.set_ylabel('FPR'); ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('phase1_comparison_fixed.png', dpi=100)
plt.show()
print('✓ Saved: phase1_comparison_fixed.png')


⚠ outputs/metadata.json not found — run the ML notebook first.
  PHASE 1 COMPARISON — CIC-IDS2017 (behaviour features, no leakage)
  Model                        F1   ROC-AUC   Recall      FPR
------------------------------------------------------------------------
  Random Forest            0.0000    0.0000   0.0000   0.0000
  Gradient Boosting        0.0000    0.0000   0.0000   0.0000
  DQN RL Agent             0.9093    0.9963   0.8411   0.0019

→ NEXT: Run Phase 2 notebook — test all three on NSL-KDD (no retraining)
✓ Saved: phase1_comparison_fixed.png


## 13 · Save Model Artefacts

In [29]:
OUT = Path('outputsRL')
OUT.mkdir(exist_ok=True)

torch.save(agent.policy_net.state_dict(), OUT / 'dqn_policy_net.pth')

with open(OUT / 'scaler_rl.pkl', 'wb') as fh:
    pickle.dump(scaler, fh)
with open(OUT / 'feature_names_rl.pkl', 'wb') as fh:
    pickle.dump(FEATURE_NAMES, fh)

rl_meta = {
    'pipeline_version': '2.0_fixed_no_leakage',
    'leakage_fix': 'source_ip, destination_ip, flow_id, source_port, timestamp removed',
    'model_type'   : 'DQN (Deep Q-Network)',
    'feature_dim'  : int(FEATURE_DIM),
    'train_rows'   : int(len(X_train_sc)),
    'test_rows'    : int(len(X_test_sc)),
    'n_episodes'   : N_EPISODES,
    'final_epsilon': round(float(agent.epsilon), 4),
    'reward_shaping': {
        'correct'       : +1.0,
        'missed_attack' : -2.0,
        'false_positive': -0.5
    },
    'performance': {
        'accuracy' : round(float(acc_rl),  4),
        'precision': round(float(prec_rl), 4),
        'recall'   : round(float(rec_rl),  4),
        'f1'       : round(float(f1_rl),   4),
        'roc_auc'  : round(float(auc_rl),  4),
        'mcc'      : round(float(mcc_rl),  4),
        'fpr'      : round(float(fpr_rl),  4)
    },
    'training_history': history
}

with open(OUT / 'rl_metadata.json', 'w') as fh:
    json.dump(rl_meta, fh, indent=2)

print('✓ Saved to outputs/')
for p in sorted(OUT.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size/1024:.1f} KB)')


✓ Saved to outputs/
  dqn_policy_net.pth  (243.9 KB)
  feature_names_rl.pkl  (1.4 KB)
  rl_metadata.json  (3.9 KB)
  scaler_rl.pkl  (2.3 KB)


## 14 · Summary & MDP Explanation for your Report

In [31]:
print('')
print('╔═══════════════════════════════════════════════════════════════╗')
print('║     DQN RL AGENT — PHASE 1 COMPLETE (FIXED, NO LEAKAGE)     ║')
print('╠═══════════════════════════════════════════════════════════════╣')
print(f'║  Accuracy        : {acc_rl:.4f}                                   ║')
print(f'║  Precision       : {prec_rl:.4f}                                   ║')
print(f'║  Recall (DR)     : {rec_rl:.4f}                                   ║')
print(f'║  F1-Score        : {f1_rl:.4f}                                   ║')
print(f'║  ROC-AUC         : {auc_rl:.4f}                                   ║')
print(f'║  False Pos. Rate : {fpr_rl:.4f}                                   ║')
print('╠═══════════════════════════════════════════════════════════════╣')
print('║  MDP FORMULATION (for your report):                          ║')
print('║  State  : scaled network flow feature vector                 ║')
print('║  Action : 0 = Allow   1 = Block                              ║')
print('║  Reward : +1 correct | -2 missed attack | -0.5 false pos.    ║')
print('║  Algo   : DQN + Experience Replay + Target Network           ║')
print('║  Epsilon: greedy decay 1.0 → 0.05 over episodes              ║')
print('╠═══════════════════════════════════════════════════════════════╣')
print('║  NEXT → Phase 2: load RF, GB, DQN → test on NSL-KDD         ║')
print('║  (no retraining — pure cross-environment generalisation)     ║')
print('╚═══════════════════════════════════════════════════════════════╝')



╔═══════════════════════════════════════════════════════════════╗
║     DQN RL AGENT — PHASE 1 COMPLETE (FIXED, NO LEAKAGE)     ║
╠═══════════════════════════════════════════════════════════════╣
║  Accuracy        : 0.9696                                   ║
║  Precision       : 0.9897                                   ║
║  Recall (DR)     : 0.8411                                   ║
║  F1-Score        : 0.9093                                   ║
║  ROC-AUC         : 0.9963                                   ║
║  False Pos. Rate : 0.0019                                   ║
╠═══════════════════════════════════════════════════════════════╣
║  MDP FORMULATION (for your report):                          ║
║  State  : scaled network flow feature vector                 ║
║  Action : 0 = Allow   1 = Block                              ║
║  Reward : +1 correct | -2 missed attack | -0.5 false pos.    ║
║  Algo   : DQN + Experience Replay + Target Network           ║
║  Epsilon: greedy decay 1.0

## 15 · What Changed vs Previous Version

| Issue | Previous | Fixed |
|---|---|---|
| `source_ip` in features | Yes (84.6% importance) | Removed |
| `destination_ip` in features | Yes | Removed |
| `flow_id`, `source_port` | Encoded as features | Removed |
| `Label` string col in X | Caused ValueError | Dropped before conversion |
| Reward for missed attack | -1.0 | **-2.0** (more realistic) |
| FPR tracked per episode | No | Yes |
| FPR in final evaluation | No | Yes |
| FPR in comparison chart | No | Yes |
| N_EPISODES | 10 | **20** (better convergence) |
| `feature_names_rl.pkl` saved | No | Yes (needed for Phase 2) |


In [33]:
# Quick convergence check — did the agent actually learn?
print('Training convergence summary:')
print(f'  Episode  1: Acc={history["accuracy"][0]:.4f}  F1={history["f1"][0]:.4f}  FPR={history["fpr"][0]:.4f}')
mid = len(history['episode']) // 2
print(f'  Episode {history["episode"][mid]:2d}: Acc={history["accuracy"][mid]:.4f}  F1={history["f1"][mid]:.4f}  FPR={history["fpr"][mid]:.4f}')
print(f'  Episode {history["episode"][-1]:2d}: Acc={history["accuracy"][-1]:.4f}  F1={history["f1"][-1]:.4f}  FPR={history["fpr"][-1]:.4f}')
print()
improvement = history['f1'][-1] - history['f1'][0]
if improvement > 0.01:
    print(f'✓ Agent converged — F1 improved by {improvement:.4f} over training')
elif improvement > 0:
    print(f'⚠ Modest improvement ({improvement:.4f}) — consider more episodes')
else:
    print(f'⚠ No improvement detected — check reward shaping or increase N_EPISODES')
print()
print('✓ Pipeline ready for Phase 2 — NSL-KDD cross-environment test')


Training convergence summary:
  Episode  1: Acc=0.9572  F1=0.8887  FPR=0.0395
  Episode 11: Acc=0.9621  F1=0.9009  FPR=0.0351
  Episode 20: Acc=0.9652  F1=0.9085  FPR=0.0322

✓ Agent converged — F1 improved by 0.0198 over training

✓ Pipeline ready for Phase 2 — NSL-KDD cross-environment test
